# Backpropagation from scratch · Module 10
Python 3 and NumPy only. Derive the equations of Lesson 38 first. The task is an image version of XOR that no linear boundary can solve, so the need for a hidden layer is visible rather than asserted. It teaches mechanics and is not a real-image benchmark.


In [1]:
import numpy as np
np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(210)
def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
def cross_entropy(z, y):
    shifted = z - z.max(axis=1, keepdims=True)
    return np.mean(np.log(np.exp(shifted).sum(axis=1)) - shifted[np.arange(len(y)), y])
A = rng.normal(0, 1, (3, 4)); B = rng.normal(0, 1, (4, 2)); x = rng.normal(0, 1, (5, 3))
assert np.allclose((x @ A) @ B, x @ (A @ B))
print('two linear layers equal one linear layer, shape', (A @ B).shape, ': stacking alone adds nothing.')
def two_bars(n, noise=.15):
    top, left = rng.integers(0, 2, n), rng.integers(0, 2, n); images = np.zeros((n, 8, 8))
    images[top == 1, 1:3, :] += 1.
    images[left == 1, :, 1:3] += 1.
    return (images + rng.normal(0, noise, images.shape)).reshape(n, -1), (top != left).astype(int)
X, y = two_bars(480)
Xtr, Xval, Xtest = X[:300], X[300:390], X[390:]; ytr, yval, ytest = y[:300], y[300:390], y[390:]
mu, scale = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-8
Xtr, Xval, Xtest = (Xtr - mu) / scale, (Xval - mu) / scale, (Xtest - mu) / scale
print('label is 1 when exactly one bar is present. Shapes:', Xtr.shape, ytr.shape)


two linear layers equal one linear layer, shape (3, 2) : stacking alone adds nothing.
label is 1 when exactly one bar is present. Shapes: (300, 64) (300,)


## Part 2: The linear baseline, and why it fails
With one bar the label is 1, and with both bars it returns to 0, so no single weight on "top bar present" can be right. Run the same softmax regression you built in Module 07 before adding anything else.


In [2]:
W = np.zeros((64, 2)); b = np.zeros(2); Y = np.eye(2)[ytr]
for _ in range(800):
    z = Xtr @ W + b; G = (softmax(z) - Y) / len(ytr)
    W -= .1 * Xtr.T @ G; b -= .1 * G.sum(axis=0)
linear_val = np.mean((Xval @ W + b).argmax(axis=1) == yval)
print('linear training loss:', cross_entropy(Xtr @ W + b, ytr), ' validation accuracy:', linear_val)
assert linear_val < .75


linear training loss: 0.5765843283772657  validation accuracy: 0.5333333333333333


## Part 3: Every backward equation, then a check
Write the shape of each array on paper: X (N,D), W1 (D,H), A1 (N,H), W2 (H,C), logits (N,C). All gradients are computed from the same forward pass, before any update. ReLU at exactly zero uses derivative zero by convention.


In [3]:
def init(hidden=32, seed=3):
    generator = np.random.default_rng(seed)
    return {'W1': generator.normal(0, .25, (64, hidden)), 'b1': np.full(hidden, .1),
            'W2': generator.normal(0, .25, (hidden, 2)), 'b2': np.zeros(2)}
def forward(X, p):
    Z1 = X @ p['W1'] + p['b1']; A1 = np.maximum(0, Z1)
    return Z1, A1, A1 @ p['W2'] + p['b2']
def loss_grad(X, y, p, weight_decay=0.):
    Z1, A1, logits = forward(X, p)
    G2 = (softmax(logits) - np.eye(2)[y]) / len(y)
    G1 = (G2 @ p['W2'].T) * (Z1 > 0)
    grads = {'W2': A1.T @ G2 + weight_decay * p['W2'], 'b2': G2.sum(axis=0),
             'W1': X.T @ G1 + weight_decay * p['W1'], 'b1': G1.sum(axis=0)}
    return cross_entropy(logits, y) + .5 * weight_decay * (np.sum(p['W1'] ** 2) + np.sum(p['W2'] ** 2)), grads
p = init(); analytic = loss_grad(Xtr[:8], ytr[:8], p, .01)[1]
for name, index in [('W1', (0, 0)), ('W1', (30, 4)), ('b1', (2,)), ('W2', (3, 1)), ('b2', (0,))]:
    value = p[name][index]; h = 1e-5
    p[name][index] = value + h; plus = loss_grad(Xtr[:8], ytr[:8], p, .01)[0]
    p[name][index] = value - h; minus = loss_grad(Xtr[:8], ytr[:8], p, .01)[0]
    p[name][index] = value
    error = abs((plus - minus) / (2 * h) - analytic[name][index])
    print('gradient check', name, index, 'error', error)
    assert error < 1e-6


gradient check W1 (0, 0) error 1.205029132034241e-12
gradient check W1 (30, 4) error 5.405537129021809e-12
gradient check b1 (2,) error 5.639127186041204e-12
gradient check W2 (3, 1) error 1.9885058877289197e-11
gradient check b2 (0,) error 8.298590981059561e-12


## Part 4: Overfit a tiny subset first
A network that cannot memorize eight examples has a bug, so this is a fast debugging check. Then train on everything with mini-batch SGD, recording training and validation evidence.


In [4]:
p = init()
for step in range(500):
    _, grads = loss_grad(Xtr[:8], ytr[:8], p)
    for name in p: p[name] -= .3 * grads[name]
tiny = loss_grad(Xtr[:8], ytr[:8], p)[0]
print('loss on 8 memorized examples:', tiny)
assert tiny < .05
p = init(); generator = np.random.default_rng(5); history = []
for epoch in range(80):
    order = generator.permutation(len(Xtr))
    for start in range(0, len(order), 32):
        chosen = order[start:start + 32]
        _, grads = loss_grad(Xtr[chosen], ytr[chosen], p, 1e-4)
        for name in p: p[name] -= .1 * grads[name]
    if epoch % 20 == 0:
        history.append((epoch, round(float(loss_grad(Xtr, ytr, p)[0]), 4), round(float(np.mean(forward(Xval, p)[2].argmax(axis=1) == yval)), 3)))
print('(epoch, training loss, validation accuracy):', history)
mlp_val = np.mean(forward(Xval, p)[2].argmax(axis=1) == yval)
print('validation accuracy, linear:', linear_val, ' MLP:', mlp_val)
assert mlp_val > linear_val + .15
print('fixed procedure, test accuracy:', np.mean(forward(Xtest, p)[2].argmax(axis=1) == ytest), ' test count:', len(ytest))


loss on 8 memorized examples: 0.00013144843235789538
(epoch, training loss, validation accuracy): [(0, 0.217, 0.822), (20, 0.0032, 1.0), (40, 0.0014, 1.0), (60, 0.0009, 1.0)]
validation accuracy, linear: 0.5333333333333333  MLP: 1.0
fixed procedure, test accuracy: 1.0  test count: 90


## Part 5: Why initialization matters (Lesson 32)
Give two hidden units identical weights and they receive identical gradients, so they stay copies forever. Then watch signal variance: if weights are too large it grows with depth, and if too small it dies. Scaling variance by one over the fan-in keeps it steady.


In [5]:
q = init(hidden=2); q['W1'][:, 1] = q['W1'][:, 0]; q['W2'][1] = q['W2'][0]; q['b1'][:] = .1
for step in range(50):
    _, grads = loss_grad(Xtr[:32], ytr[:32], q)
    for name in q: q[name] -= .1 * grads[name]
assert np.allclose(q['W1'][:, 0], q['W1'][:, 1])
print('after 50 steps the two hidden units are still identical.')
def variance_through_depth(weight_std, depth=10, fan_in=256):
    x = rng.normal(0, 1, (400, fan_in))
    for layer in range(depth):
        x = x @ rng.normal(0, weight_std, (fan_in, fan_in))
    return x.var()
for name, std in [('too small', .5 / 256 ** .5), ('scaled by fan-in', 1 / 256 ** .5), ('too large', 2 / 256 ** .5)]:
    print(f'{name:18s} output variance after 10 layers: {variance_through_depth(std):.3g}')
assert variance_through_depth(.5 / 256 ** .5) < .01 < 100 < variance_through_depth(2 / 256 ** .5)


after 50 steps the two hidden units are still identical.
too small          output variance after 10 layers: 9.43e-07
scaled by fan-in   output variance after 10 layers: 1.03
too large          output variance after 10 layers: 1.09e+06


## Part 6: Convolution shapes and receptive fields (Lesson 35)
Now that you know layers, count them. Check the output-size formula against a direct count of kernel positions. Then see why stacking small kernels with stride widens the view quickly. A fully connected layer on the same image needs far more weights, because a convolution shares its kernel across positions.


In [6]:
def output_size(n, kernel, padding, stride): return (n + 2 * padding - kernel) // stride + 1
def count_positions(n, kernel, padding, stride): return len(range(0, n + 2 * padding - kernel + 1, stride))
for n, k, pad, s in [(7, 3, 1, 2), (32, 3, 1, 1), (32, 5, 2, 2), (28, 2, 0, 2)]:
    assert output_size(n, k, pad, s) == count_positions(n, k, pad, s)
    print(f'input {n}, kernel {k}, padding {pad}, stride {s}: output {output_size(n, k, pad, s)}')
def parameters(kernel, channels_in, channels_out): return kernel * kernel * channels_in * channels_out + channels_out
print('3x3 layer, 3 to 8 channels:', parameters(3, 3, 8), 'parameters')
print('fully connected from a 32x32x3 image to 8 outputs:', 32 * 32 * 3 * 8 + 8, 'parameters')
assert parameters(3, 3, 8) == 224
def receptive_field(layers):
    r, j = 1, 1
    for kernel, stride in layers:
        r += (kernel - 1) * j; j *= stride
    return r, j
print('(receptive field, jump) for three 3x3 stride-1 layers:', receptive_field([(3, 1)] * 3))
print('for four 3x3 stride-2 layers:', receptive_field([(3, 2)] * 4))
assert receptive_field([(3, 1)] * 3) == (7, 1)


input 7, kernel 3, padding 1, stride 2: output 4
input 32, kernel 3, padding 1, stride 1: output 32
input 32, kernel 5, padding 2, stride 2: output 16
input 28, kernel 2, padding 0, stride 2: output 14
3x3 layer, 3 to 8 channels: 224 parameters
fully connected from a 32x32x3 image to 8 outputs: 24584 parameters
(receptive field, jump) for three 3x3 stride-1 layers: (7, 1)
for four 3x3 stride-2 layers: (31, 16)


## Your experiment
Remove the ReLU and predict the validation accuracy before you run it. Then declare one comparison on a fresh held-out split: learning rate, hidden size or weight decay. Change one factor at a time and report the settings that failed. Your training report should include your gradient checks and explain why this result says nothing about real images. Practical Lab 04 is the next step.
